In [ ]:
# CELL 1 — Board Setup and Overlay Load
from pynq import Overlay, allocate
import numpy as np
import time

print("Loading ECG CNN overlay...")
ol = Overlay('/home/xilinx/jupyter_notebooks/ecg_cnn.bit')
print("Overlay loaded successfully!")
print()

print(ol.ip_dict.keys())

cnn = ol.ecg_cnn_top_0

print("IP blocks available:")
for name in ol.ip_dict:
    print(f"  {name}")

print()
print("CNN IP registers:")
print("  CTRL        @ 0x00")
print("  input_r_1   @ 0x10")
print("  input_r_2   @ 0x14")
print("  output_r_1  @ 0x1c")
print("  output_r_2  @ 0x20")


In [ ]:
# CELL 2 — Allocate DMA Buffers
INPUT_LEN = 186
NUM_CLASSES = 5

in_buf  = allocate(shape=(INPUT_LEN,), dtype=np.float32)
out_buf = allocate(shape=(NUM_CLASSES,), dtype=np.float32)

print(f"Input buffer: shape={in_buf.shape} dtype={in_buf.dtype}")
print(f"  Physical address: 0x{in_buf.physical_address:08X}")
print(f"Output buffer: shape={out_buf.shape} dtype={out_buf.dtype}")
print(f"  Physical address: 0x{out_buf.physical_address:08X}")
print()
print("Buffers allocated in DDR — CNN can DMA directly to/from these addresses.")


In [ ]:
# CELL 3 — Inference Function
CLASS_NAMES  = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unknown']
CLASS_SHORT  = ['N', 'S', 'V', 'F', 'Q']

def run_inference(ecg_beat):
    beat = np.array(ecg_beat, dtype=np.float32).flatten()
    assert len(beat) == INPUT_LEN, f"Expected 186 samples, got {len(beat)}"

    in_buf[:] = beat
    in_buf.flush()

    cnn.write(0x10, in_buf.physical_address & 0xFFFFFFFF)
    cnn.write(0x14, (in_buf.physical_address >> 32) & 0xFFFFFFFF)
    cnn.write(0x1C, out_buf.physical_address & 0xFFFFFFFF)
    cnn.write(0x20, (out_buf.physical_address >> 32) & 0xFFFFFFFF)

    t_start = time.perf_counter()
    cnn.write(0x00, 1)

    timeout = 5.0
    t_deadline = t_start + timeout

    while True:
        status = cnn.read(0x00)
        if status & 0x2:
            break
        if time.perf_counter() > t_deadline:
            raise TimeoutError("CNN inference timed out")
        time.sleep(0.0001)

    t_end = time.perf_counter()
    latency_ms = (t_end - t_start) * 1000.0

    out_buf.invalidate()
    probs = np.array(out_buf, dtype=np.float32).copy()

    predicted_class = int(np.argmax(probs))
    return predicted_class, probs, latency_ms

print("run_inference() function defined.")


In [ ]:
# CELL 4 — Test with Synthetic Beat
print("="*55)
print("SANITY CHECK — Synthetic zero beat")
print("="*55)

test_beat = np.zeros(186, dtype=np.float32)

pred, probs, latency = run_inference(test_beat)

print("Input : flat zero signal")
print("Output probabilities:")

for i, (name, p) in enumerate(zip(CLASS_NAMES, probs)):
    bar = "#" * int(p * 40)
    print(f"{CLASS_SHORT[i]} ({name:<18}): {p:.4f} {bar}")

print(f"Predicted class : {CLASS_SHORT[pred]} — {CLASS_NAMES[pred]}")
print(f"Inference time  : {latency:.3f} ms")

if latency < 100:
    print("✓ Latency OK")
else:
    print("⚠ Latency high")


In [ ]:
# CELL 5 — Load Real Test Data
import os

X_PATH = '/home/xilinx/jupyter_notebooks/X_pynq.npy'
Y_PATH = '/home/xilinx/jupyter_notebooks/y_pynq.npy'

if not os.path.exists(X_PATH):
    print("Test data not found — upload files first")
else:
    X_test = np.load(X_PATH)
    y_test = np.load(Y_PATH)

    print(f"Loaded: X_test={X_test.shape}, y_test={y_test.shape}")

    print("="*55)
    print("RUNNING INFERENCE ON ALL TEST SAMPLES")
    print("="*55)

    y_pred = []
    latencies = []

    for i in range(len(y_test)):
        pred, probs, lat = run_inference(X_test[i])
        y_pred.append(pred)
        latencies.append(lat)

        if (i+1) % 50 == 0:
            print(f"{i+1}/{len(y_test)} done ...")

    y_pred = np.array(y_pred)

    accuracy = np.mean(y_pred == y_test) * 100

    print("\nRESULTS")
    print(f"Overall accuracy: {accuracy:.2f}%")

    print("\nPer-class accuracy:")
    for c in range(5):
        mask = (y_test == c)
        correct = np.sum((y_pred == c) & mask)
        total = np.sum(mask)
        acc = correct / total * 100 if total > 0 else 0
        print(f"{CLASS_NAMES[c]:<20} {acc:.2f}%")

    lat_arr = np.array(latencies)
    print("\nLatency stats:")
    print(f"Mean : {lat_arr.mean():.3f} ms")
    print(f"Min  : {lat_arr.min():.3f} ms")
    print(f"Max  : {lat_arr.max():.3f} ms")


In [ ]:
# CELL 6 — Confusion Matrix
if 'y_pred' in dir() and len(y_pred) > 0:
    print("Confusion Matrix (rows=actual, cols=predicted)\n")

    header = " " * 18 + " ".join(f"{s:>6}" for s in CLASS_SHORT)
    print(header)
    print("-" * (18 + 6*5))

    for actual in range(5):
        row = f"{CLASS_NAMES[actual]:>18}"
        for pred_c in range(5):
            count = int(np.sum((y_test == actual) & (y_pred == pred_c)))
            row += f"{count:6d}"
        print(row)

    print("\nDiagonal = correct predictions")


In [ ]:
# CELL 7 — Power Measurement
try:
    from pynq.pmbus import get_rails

    rails = get_rails()
    print("Power Rails on PYNQ-Z2:\n")

    total_power = 0.0

    for name, rail in rails.items():
        try:
            v = rail.voltage
            i = rail.current
            p = v * i
            total_power += p
            print(f"{name:<15} {v:.3f}V {i:.3f}A {p:.3f}W")
        except:
            print(f"{name:<15} reading unavailable")

    print(f"\nTOTAL POWER: {total_power:.3f} W")

except ImportError:
    print("pynq.pmbus not available")
